# LSTM Tabanlı Regresyon ile Hava Sıcaklığı Tahmini
### Yüksek lisans vize sunumu · Jena Climate · TensorFlow/Keras

**Problem türü:** Denetimli öğrenme — **regresyon**. Çıktı, sürekli sayısal hava sıcaklığıdır (°C).

**Kullanılan tek yaklaşım:** LSTM tabanlı sinir ağı regresyonu. LSTM bir ağ mimarisidir; regresyon oluşu sayısal hedefi ve doğrusal çıkış katmanından kaynaklanır.

**Araştırma sorusu:** Son 24 saatin 5 meteorolojik değişkeninden bir saat sonraki sıcaklığı tahmin edebilir miyiz?

**Sunum akışı:** Regresyon problemi → Veri → Ön işleme → LSTM regresyon mimarisi → Kayıp fonksiyonu → Eğitim → MAE/RMSE/R² → Sınırlamalar.

> Sunum öncesi `python -m src.train --config config.yaml` çalıştırın. Bu notebook eğitim sonucunu okur, kendisi eğitmez.

## 1. Regresyon problemi nedir, burada nasıl uygulanır?

**Regresyon** bağımlı değişkenin sürekli sayısal değerini giriş özelliklerinden tahmin etmektir. Sınıflandırmada “soğuk/ılık/sıcak” gibi etiketler olurdu; burada doğrudan ör. **12.7 °C** tahmin edilir.

- **Girdi X:** 24 geçmiş saat × 5 meteorolojik özellik (basınç, sıcaklık, nem, rüzgâr hızı, yönü).
- **Hedef y:** Bir saat sonrasının sıcaklığı, `T (degC)`.
- **Fonksiyon:** $f_\theta: \mathbb{R}^{24\times5} \to \mathbb{R}$; öğrenilen ağırlıklar $\theta$.
- **Yöntem:** Sıralı gözlemleri işlemek için LSTM, bir sayısal tahmin üretmek için aktivasyonsuz `Dense(1)`.
- **Kapsam:** Sadece bu LSTM regresyon yöntemi eğitilir.

**Neden LSTM?** Girdiler zaman sırasına sahiptir. LSTM geçmişe dair durum bilgisi taşıyarak gecikmeli ilişkileri öğrenebilir; başarı garantisi değildir.

## 2. Veri seti: Jena Climate

- Kaynak: Max Planck Institute for Biogeochemistry / TensorFlow veri dağıtımı.
- Orijinal ölçümler yaklaşık 10 dakika aralıklıdır; her 6. satır alınarak saatlik örnek kullanılır.
- Kullanılan özellikler: basınç, sıcaklık, bağıl nem, rüzgâr hızı, rüzgâr yönü.
- `T (degC)` hem geçmiş giriş özelliği hem gelecekteki hedef değişkendir.

**Neden bu veri?** Gerçek meteorolojik ölçümler, indirilebilir veri ve belirgin zamansal örüntüler.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import yaml
import matplotlib.pyplot as plt

# Notebook, repo kökünden veya notebooks/ klasöründen açılabilir.
ROOT = Path.cwd()
if not (ROOT / 'config.yaml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'config.yaml').exists(), 'Notebooku proje klasöründen açın.'
config = yaml.safe_load((ROOT / 'config.yaml').read_text(encoding='utf-8'))
config

## 3. Keşifsel veri analizi

Veri setinin zaman sırasını, temel istatistiklerini ve sıcaklığın zaman içinde nasıl değiştiğini inceliyoruz. Bu kısım eğitim ve test arasında **gelecek bilgisini öğrenmek** için kullanılmaz; yalnızca tanıtım/keşif içindir.

In [ ]:
csv_path = ROOT / 'data/raw' / config['data']['csv_name']
if not csv_path.exists():
    raise FileNotFoundError('CSV yok. Önce: python -m src.train --config config.yaml')
df = pd.read_csv(csv_path)
print('Ham veri boyutu:', df.shape)
display(df.head())
display(df[config['data']['features']].describe().round(2))
ax = df.iloc[::6].iloc[:24*14].plot(y='T (degC)', figsize=(12, 3), legend=False)
ax.set(title='Örnek: İlk iki haftanın saatlik sıcaklığı', xlabel='Saatlik örnek', ylabel='°C')
plt.show()

## 4. Ön işleme ve veri sızıntısını önleme

1. Geçersiz rüzgâr değeri `-9999` → eksik değer → ileri doldurma.
2. **Zamansal %70 / %20 / %10** eğitim, doğrulama, test bölmesi.
3. **StandardScaler yalnız eğitim** verisine fit edilir.
4. Kaydırmalı pencere: 24 saatlik giriş, izleyen 1 saatlik hedef.

**Kritik nokta:** Rastgele bölme yapılmaz. Doğrulama/test hedefleri ayrı tutulur; pencere girdisinin daha erken tarihlerden gelmesine izin verilir.

## 5. LSTM tabanlı regresyon mimarisi

LSTM gizli durum $h_t$ ve hücre belleği $c_t$ tutar; unutma, giriş ve çıkış kapıları zaman içinde bilgi akışını düzenler.

**Mimari:** `Input(24,5) → LSTM(32) → Dropout(0.2) → Dense(1, activation=linear)`.

- `LSTM(32)`: Geçmiş 24 saatin zamansal özetini öğrenir.
- `Dropout(0.2)`: Eğitimde aşırı uyumu azaltmaya yardımcı olur.
- `Dense(1)`: **Tek ve sürekli sayısal** regresyon çıktısı üretir.
- **Neden Softmax/Sigmoid değil?** Bunlar sınıflandırma/olasılık tipi kısıtlar getirir; °C çıktısı negatif ve pozitif olabilir.
- **Adam + MSE:** Ağırlıklar, karesel regresyon kaybını minimize etmek için güncellenir.

**Not:** LSTM kendi başına geleneksel doğrusal regresyon değildir; regresyon görevinde kullanılan bir sinir ağı mimarisidir.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from src.models import build_lstm
m = build_lstm(lookback=config['data']['lookback'],
               num_features=len(config['data']['features']),
               units=config['model']['units'],
               dropout=config['model']['dropout'],
               learning_rate=config['training']['learning_rate'])
m.summary()

## 6. Regresyon kaybı ve deney düzeni

Eğitimde kullanılan hedef sıcaklık `StandardScaler` ile standartlaştırılır. Model bu ölçek üzerinde **MSE** değerini minimize eder.

$$\mathrm{MSE}=\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat y_i)^2$$

Büyük hatalar kareleri alındığı için daha yüksek cezalandırılır. Adam optimizer ağırlıkları günceller. Doğrulama MSE iyileşmediğinde `EarlyStopping` durdurur, `ModelCheckpoint` en iyi modeli kaydeder.

**Önemli:** Test verisi eğitim veya model seçimi için kullanılmaz. Tek regresyon yaklaşımı olan LSTM değerlendirilir; ikinci algoritma eğitilmez.

## 7. Eğitim grafiği ve gerçek tahminler

Aşağıdaki grafikler **gerçek eğitim çalıştırıldıktan sonra** `outputs/figures/` klasöründe oluşturulur; notebook bunları okur.

In [ ]:
from IPython.display import display, Image
for name in ['training_history.png', 'predictions.png']:
    path = ROOT / 'outputs/figures' / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f'Üretilmemiş sonuç: {name}. Önce eğitim çalıştırılmalı.')

## 8. Regresyon başarısının ölçülmesi

**MAE:** $\frac1n\sum |y_i-\hat y_i|$ — ortalama mutlak hata; **°C**.

**RMSE:** $\sqrt{\frac1n\sum (y_i-\hat y_i)^2}$ — büyük hatalara daha duyarlı; **°C**.

**R²:** $1-\frac{\sum (y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}$ — boyutsuz; 1 mükemmele yakın, 0 sabit ortalama tahminiyle aynı hata, negatif değer de mümkündür.

Ölçümler test sıcaklıkları özgün **°C** birimine geri çevrildikten sonra hesaplanır. Sonuçlar yalnızca gerçekten eğitilmiş modelden okunur.

In [ ]:
metrics_path = ROOT / 'outputs/metrics.json'
if metrics_path.exists():
    scores = json.loads(metrics_path.read_text(encoding='utf-8'))
    if 'lstm' not in scores:
        raise ValueError('LSTM regresyon metrikleri bulunamadı.')
    print('LSTM regresyonu — test sonuçları')
    display(pd.DataFrame([scores['lstm']], index=['LSTM Regresyon']).round(4))
else:
    print('Gerçek test sonucu henüz yok. Önce python -m src.train komutunu çalıştırın.')

## 9. Sonuçların yorumu ve sınırlamalar

**MAE/RMSE küçükse:** Test sıcaklıklarını daha az hata ile tahmin ettiğimizi gösterir; sonuçlar °C olarak açıklanır.

**R² yüksekse:** Test kümesinde sabit ortalama öngörüsüne göre hata azalır; nedensellik veya başka istasyonlarda aynı başarı anlamına gelmez.

**Hatalar büyükse:** Bu da raporlanması gereken gerçek bir bulgudur. Tek model üzerinde çalışmak başarının yüksek olmasını garanti etmez.

**Sınırlamalar:** Tek istasyon; tek adımlı sıcaklık tahmini; rüzgâr yönü sin/cos ile kodlanmıyor; tek tarih bölmesi; farklı seed / dönem denemeleri yok.

**Dikkat:** “LSTM doğrusal regresyon algoritmasıdır” demek yanlıştır. Doğru tanım: **LSTM tabanlı sinir ağı regresyonu**.

## 10. Özet, kaynaklar ve olası sözlü sorular

**Özet:** 24 saatlik 5 meteorolojik girdiyle tek bir sürekli değişkeni (1 saat sonraki °C sıcaklık) tahmin eden **LSTM regresyon modeli** kuruldu. MSE ile optimize edildi ve MAE, RMSE, R² ile değerlendirilir.

**Hocanın sorabileceği sorular**

- **Bu neden regresyon?** Çıktı sınıf değil, reel sayı (°C).
- **LSTM regresyon yöntemi midir?** LSTM mimarisidir; son doğrusal katman ve sürekli hedefle regresyon için kullanılır.
- **Neden MSE?** Türevlenebilir, büyük hataları daha fazla cezalandırır.
- **Neden R² yeterli değil?** Hata büyüklüğünü °C ile göstermez; MAE ve RMSE de gerekir.
- **Neden random split yok?** Zaman serisinde gelecek bilgisi sızabilir.

**Kaynaklar**
1. Hochreiter & Schmidhuber (1997), *Long Short-Term Memory*. https://doi.org/10.1162/neco.1997.9.8.1735
2. TensorFlow, *Time series forecasting tutorial*. https://www.tensorflow.org/tutorials/structured_data/time_series
3. [Jena Climate Dataset](https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip)